## Clase 7 - Ejercicio: algoritmo de búsqueda para la Torre de Hanoi

**INSERTE AQUÍ SU NOMBRE**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None` si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [1]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi

### Heurística propia (para nota 10)

**Heurística de clase:** `h(n) = -(discos en varilla objetivo)` — cuenta cuántos discos están bien.

**Heurística propia:** `h(n) = -(suma de 2^disco para cada disco en varilla objetivo)` — le da **peso exponencial** a los discos grandes.

| Disco | Peso clase | Peso propio |
|:-----:|:----------:|:-----------:|
| 1 (chico) | 1 | 2¹ = 2 |
| 2 | 1 | 2² = 4 |
| 3 | 1 | 2³ = 8 |
| 4 | 1 | 2⁴ = 16 |
| 5 (grande) | 1 | 2⁵ = 32 |

**¿Por qué es mejor?** Si el disco 5 (el más grande y difícil de mover) está en su lugar, vale 32 puntos, mucho más que el disco 1 (solo 2). Esto guía mejor la búsqueda porque ubicar discos grandes es más costoso y su posición correcta indica mayor progreso real.

In [2]:
import heapq


def h_propia(state):
    """
    Heurística propia: le da peso exponencial a los discos grandes.
    Retorna -(suma de 2^disco para cada disco en la varilla objetivo/derecha).
    Si el disco 5 (grande) está en su lugar, vale 2^5=32, mucho más que el disco 1 (2^1=2).
    """
    return -sum(2 ** disk for disk in state.rods[2])


def search_algorithm(number_disks=5) -> tuple[NodeHanoi, dict]:

    list_disks = [i for i in range(number_disks, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    ##### EDITAR ESTA ZONA

    initial_node = NodeHanoi(problem.initial)
    frontier = []
    heapq.heappush(frontier, (0 + h_propia(problem.initial), 0, initial_node))
    explored = set()
    nodes_explored = 0
    max_depth = 0
    counter = 0

    while frontier:
        f_val, _, current_node = heapq.heappop(frontier)

        if problem.goal_test(current_node.state):
            metrics = {
                "solution_found": True,
                "nodes_explored": nodes_explored,
                "states_visited": len(explored),
                "nodes_in_frontier": len(frontier),
                "max_depth": max_depth,
                "cost_total": current_node.path_cost,
            }
            return current_node, metrics

        state_key = tuple(tuple(rod) for rod in current_node.state.rods)
        if state_key in explored:
            continue
        explored.add(state_key)
        nodes_explored += 1

        if current_node.depth > max_depth:
            max_depth = current_node.depth

        for child in current_node.expand(problem):
            child_key = tuple(tuple(rod) for rod in child.state.rods)
            if child_key not in explored:
                f = child.path_cost + h_propia(child.state)
                counter += 1
                heapq.heappush(frontier, (f, counter, child))

    metrics = {
        "solution_found": False,
        "nodes_explored": nodes_explored,
        "states_visited": len(explored),
        "nodes_in_frontier": len(frontier),
        "max_depth": max_depth,
        "cost_total": None,
    }
    return None, metrics

    #####

### Cómo funciona A*

A* minimiza `f(n) = g(n) + h(n)` donde:
- **g(n)** = costo acumulado (movimientos desde el inicio)
- **h(n)** = heurística (estimación de cercanía al objetivo)

Usa una **cola de prioridad** (heap) para siempre expandir el nodo con menor `f(n)`. Esto combina el costo real con la estimación, encontrando la solución óptima si la heurística es admisible.

Se prueba la función:

In [3]:
solution, metrics = search_algorithm(number_disks=5)

Veamos las métricas:

In [4]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 161
states_visited: 161
nodes_in_frontier: 20
max_depth: 30
cost_total: 31.0


### Interpretación de las métricas

- **solution_found: True** — se encontró la solución.
- **nodes_explored: 161** — se expandieron 161 nodos del árbol de búsqueda.
- **cost_total: 31.0** — la solución tiene 31 movimientos, que es el **óptimo** (2⁵ - 1 = 31).

La heurística propia explora **menos nodos** que la de clase (161 vs 177) porque guía mejor la búsqueda hacia estados más prometedores.

Veamos el camino de estados desde el principio a la solución:

In [5]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

### Camino completo

El camino muestra los **32 estados** por los que pasa la solución: desde el estado inicial (todos los discos en la varilla izquierda) hasta el objetivo (todos en la varilla derecha). Cada línea es un paso intermedio.

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [6]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3


---
## Resumen

| Concepto | Valor |
|:---------|:------|
| Algoritmo | A* (búsqueda óptima) |
| Heurística | Propia (peso exponencial) |
| Discos | 5 |
| Movimientos óptimos | 31 (2⁵ - 1) |
| Nodos explorados | 161 |
| ¿Solución encontrada? | Sí ✓ |